#### Стеганография аудио и видео информации. Лабораторная работа №2
|   Группа          |   ФИО             |   
|   :------------:  |   :------------:  |
|   М092501(71)     |   Шарибжанов И.Т. |

**Цель работы:**
Изучение метода скрытия текстовой информации методом изменения интервала между словами, получение навыков реализации изученного метода

In [9]:
# Вспомогательные функции для работы с двоичным кодом
import os

def text_to_bits(text):
    """Преобразует строку в строку из нулей и единиц (8 бит на символ)."""
    # Используем utf-8, чтобы корректно обрабатывать русские буквы (Фамилию)
    bytes_data = text.encode('utf-8')
    return ''.join(format(b, '08b') for b in bytes_data)

def bits_to_text(bits):
    """Преобразует строку из битов обратно в текст."""
    # Разбиваем биты на группы по 8 (байты)
    byte_array = bytearray(int(bits[i:i+8], 2) for i in range(0, len(bits), 8))
    return byte_array.decode('utf-8', errors='ignore')

In [10]:
# Функция встраивания (Шифратор)
def encrypt(msg_file, container_file, stego_file):
    # Чтение файлов сообщения и контейнера
    with open(msg_file, 'r', encoding='utf-8') as f:
        message = f.read()
    with open(container_file, 'r', encoding='utf-8') as f:
        container = f.read()
        
    # Преобразуем сообщение в биты
    msg_bits = text_to_bits(message)
    
    # Разбиваем контейнер на слова по пробелам
    words = container.split(' ')
    num_intervals = len(words) - 1
    
    # Проверяем, достаточно ли интервалов для скрытия
    if len(msg_bits) > num_intervals:
        print(f"Ошибка: Количество интервалов ({num_intervals}) недостаточно для скрытия информации ({len(msg_bits)} бит)!")
        return False
        
    print(f"Количество интервалов в контейнере {num_intervals}")
    print(f"Количество бит в сообщении: {len(msg_bits)}")
    
    stego_text = ""
    
    # Формируем стеганоконтейнер
    for i in range(len(words)):
        stego_text += words[i]
        
        # Если это не последнее слово, добавляем пробелы
        if i < len(words) - 1:
            if i < len(msg_bits):
                # Если нужно вставить 1 -> 1 пробел, если 0 -> 2 пробела
                if msg_bits[i] == '1':
                    stego_text += " "
                else:
                    stego_text += "  "
            else:
                # Если биты закончились, оставляем стандартный 1 пробел
                stego_text += " "
                
    # Записываем результат в файл стеганоконтейнера
    with open(stego_file, 'w', encoding='utf-8') as f:
        f.write(stego_text)
        
    print(f"Работа завершена. Результаты сохранены в {stego_file}")
    return len(msg_bits) # Возвращаем количество встроенных бит для дешифратора

In [11]:
# Функция извлечения (Дешифратор)
def decrypt(stego_file, output_file, num_bits):
    with open(stego_file, 'r', encoding='utf-8') as f:
        stego_text = f.read()
        
    extracted_bits = ""
    i = 0
    bits_found = 0
    
    # Ищем биты сообщения, сравнивая элементы с пробелом
    while i < len(stego_text) and bits_found < num_bits:
        if stego_text[i] == ' ':
            # Проверяем следующий символ
            if i + 1 < len(stego_text) and stego_text[i+1] == ' ':
                extracted_bits += '0' # Два пробела = 0
                i += 2 # Пропускаем оба пробела
            else:
                extracted_bits += '1' # Один пробел = 1
                i += 1
            bits_found += 1
        else:
            i += 1
            
    # Переводим группы по 8 бит в символы
    secret_message = bits_to_text(extracted_bits)
    
    # Сохраняем результат
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(secret_message)
        
    print(f"Извлеченное сообщение: {secret_message}")
    print(f"Результаты дешифровки сохранены в {output_file}")

In [12]:
# 1. Создаем исходные файлы для лабораторной работы
my_surname = "Шарибжанов"
container_text = """Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.
Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.
Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.
"""

with open('A.txt', 'w', encoding='utf-8') as f:
    f.write(my_surname)
    
with open('B.txt', 'w', encoding='utf-8') as f:
    f.write(container_text)

print("--- ЭТАП 1: СКРЫТИЕ ДАННЫХ ---")
# Прячем данные из A.txt в B.txt и сохраняем в C.txt 
hidden_bits_count = encrypt('A.txt', 'B.txt', 'C.txt')

print("\n--- ЭТАП 2: ИЗВЛЕЧЕНИЕ ДАННЫХ ---")
# Расшифровываем данные из C.txt и сохраняем в decrypted_A.txt
if hidden_bits_count:
    decrypt('C.txt', 'decrypted_A.txt', hidden_bits_count)

--- ЭТАП 1: СКРЫТИЕ ДАННЫХ ---
Количество интервалов в контейнере 204
Количество бит в сообщении: 160
Работа завершена. Результаты сохранены в C.txt

--- ЭТАП 2: ИЗВЛЕЧЕНИЕ ДАННЫХ ---
Извлеченное сообщение: Шарибжанов
Результаты дешифровки сохранены в decrypted_A.txt
